## Сетап

Данный блокнот - продолжение базового

Здесь с помощью дополнительных данных о преимуществе сторон я пытаюсь увеличить `score` на тесте на метрике:

$$ \text{Gini} = 2 \cdot \text{ROC-AUC} - 1$$

Конечная цель — внедрить в пайплайн новую фичу, увилисить `gini`

## Импорты и настройка

In [1]:
import sys
import os
import pandas as pd
import numpy as np
import joblib

from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

sys.path.append(os.path.abspath('..'))

def gini(y_true, y_score):
    return 2 * roc_auc_score(y_true, y_score) - 1.0

def sqrt_transformer(X):
    return np.sqrt(X)

def add_missing_flag(X):
    return X.isna().astype(float)

## Основная часть

### 1.1. Агрегации

В данных есть большой кусок про advantage — преимущество команды сил Света с точностью до минуты, по золоту и опыту, всё так же **в пределах 15 минут**. Лежат они в `dota_adv.csv`. Поработаю с ним

In [3]:
dota_adv = pd.read_csv('../data/raw/dota_adv.csv')
dota_adv.head()

,match_id,radiant_gold_adv,radiant_exp_adv
0,526846,[ 0 159 452 1904 2100 3290 3290 3290 3290 ...,[ 0 68 658 1397 1435 2118 2118 1923 1923 ...
1,511496,[ 0 -151 -141 12 -165 -151 -151 4 377 ...,[ 0 1 -136 243 -270 -8 -8 -169 -169 ...
2,90272,[],[]
3,153647,[],[]
4,694826,[],[]


In [4]:
def parse_adv(series):
    clean_series = series.str.strip('[]').str.replace(r'\s+', ' ', regex=True)
    clean_series = [np.fromstring(x, sep=' ', dtype=int).tolist() 
          for x in clean_series.astype(str)]
    result = np.array([xi + [np.nan] *(16 - len(xi)) for xi in clean_series])
    return result

gold_series = parse_adv(dota_adv['radiant_gold_adv'])
exp_series = parse_adv(dota_adv['radiant_exp_adv'])

Для начала возьмём простые агрегации: среднее, стандартное отклонение, доля минут, в которых radiant в преимуществе и последнее значение (на 15 минуте если есть)

In [5]:
mean_gold_agg = pd.Series(np.mean(gold_series, axis=1))
mean_exp_agg = pd.Series(np.mean(exp_series, axis=1))

std_gold_agg = pd.Series(np.std(gold_series, axis=1))
std_exp_agg = pd.Series(np.std(exp_series, axis=1))

frac_gold_agg = pd.Series([np.count_nonzero(i > 0) / np.size(i) for i in gold_series])
frac_exp_agg = pd.Series([np.count_nonzero(i > 0) / np.size(i) for i in exp_series])

last_gold_agg = pd.Series([i[-1] for i in gold_series])
last_exp_agg = pd.Series([i[-1] for i in exp_series])

dota_adv['mean_radiant_gold_adv'] = mean_gold_agg.astype(np.float64)
dota_adv['mean_radiant_exp_adv'] = mean_exp_agg.astype(np.float64)
dota_adv['std_radiant_gold_adv'] = std_gold_agg.astype(np.float64)
dota_adv['std_radiant_exp_adv'] = std_exp_agg.astype(np.float64)
dota_adv['frac_radiant_gold_adv'] = frac_gold_agg.astype(np.float64)
dota_adv['frac_radiant_exp_adv'] = frac_exp_agg.astype(np.float64)
dota_adv['last_radiant_gold_adv'] = last_gold_agg.astype(np.float64)
dota_adv['last_radiant_exp_adv'] = last_exp_agg.astype(np.float64)

dota_adv = dota_adv.drop(columns=['radiant_gold_adv', 'radiant_exp_adv'])
dota_adv = dota_adv.fillna(0)

Тут по логике из базового блокнота пишем `AdvantageEncoder`, который будет принимать матчи и возвращать наши агрегации по ним. Я его написал в `../src/transformers.py`. Протестим

In [6]:
matches_df_train = pd.read_csv('../data/raw/matches_df_train.csv', parse_dates=['date'])
matches_df_train = matches_df_train.sort_values(by=['date']).reset_index().drop(columns='index')
matches_df_test = pd.read_csv('../data/raw/matches_df_test.csv')

X_train, X_val, y_train, y_val = train_test_split(
    matches_df_train.drop(columns=['radiant_win']), matches_df_train['radiant_win'], shuffle=False, test_size=0.2
)

Возьмём наш предыдущий пайплайн и добавим новый трансформер в блок препроцессинга

In [7]:
from src.transformers import AdvantageEncoder

pipeline = joblib.load('../models/base_pipeline.pkl')
current_transformers = list(pipeline.named_steps['preprocessor'].transformers)
adv_pipeline = Pipeline([
    ('encoder', AdvantageEncoder(dota_adv)),
    ('scaler', StandardScaler())
])
new_step = ('advantages_prep', adv_pipeline, ['match_id'])
current_transformers.append(new_step)
pipeline.named_steps['preprocessor'].transformers = current_transformers

Пайплайн изменён, но дело в том, что в нём старая модель со старыми гиперпараметрами, а у нас уже новые признаки. **Оптимизируем** гиперпараметры для нового пайплайна с `optuna`

In [ ]:
import optuna

def objective(trial):
    global pipeline, X_train, X_val, y_train, y_val

    params = {
        'C': trial.suggest_float('C', 1e-5, 1000, log=True),
        'penalty': trial.suggest_categorical('penalty', ['l2']),
        'solver': 'lbfgs',
        'max_iter':  trial.suggest_int('max_iter', 1000, 2000, 100),
    }

    pipeline.set_params(classifier__C=params['C'], classifier__penalty=params['penalty'],
                        classifier__solver=params['solver'], classifier__max_iter=params['max_iter'])
    
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict_proba(X_val)[:, 1]
    score = gini(y_val, y_pred)
    return score

study = optuna.create_study(direction='maximize')
study.optimize(objective, show_progress_bar=True, n_trials=25)

Меняем модель на новую с оптимизированными параметрами

In [ ]:
best_params = study.best_params
pipeline.steps[-1] = ('classifier', LogisticRegression(**best_params))
pipeline.fit(X_train, y_train)

In [16]:
y_pred_train = pipeline.predict_proba(X_train)[:, 1]
y_pred_val = pipeline.predict_proba(X_val)[:, 1]

print(f'Gini на трейне: {gini(y_train, y_pred_train):.2f}\nGini на валидации: {gini(y_val, y_pred_val):.2f}')

Gini на трейне: 0.39
Gini на валидации: 0.39


Локальный предикт воодушевляет. Проверим на Kaggle

In [18]:
y_pred_test = pipeline.predict_proba(matches_df_test)[:, 1]
test_predictions = pd.DataFrame({
    'ID': matches_df_test['match_id'],
    'Value': y_pred_test
})
test_predictions.to_csv('../data/final/advanced_test_predictions.csv', index=False)

Предикт выдал значение 0.36983 на тесте в Kaggle. Предыдущая посылка была 0.26734

**Прирост огромный**

### 1.2. Тренд

Попробуем собрать агрегацию похитрее - она будет обозначать тренд, который есть в графиках преимущества, и если пословица верна, наша модель уловит эту зависимость. Подобная агрегация может помочь модели предсказать будущие значения преимуществ

Делать это я буду трансформером со следующим функционалом

1. Принимает на вход функцию колонку
2. Выделяет коэффициент наклона (`slope`, он же $`\alpha`$) при помощи МНК $(X^TX)^{-1}X^Ty$
3. Считает `r2` и `intercept` для одного advantage (тоже фичи)

Трансформер `TrendTransformer` я написал и сохранил там же - `../src/transformers.py`. Протестим

In [8]:
from src.transformers import TrendTransformer

current_transformers = list(pipeline.named_steps['preprocessor'].transformers)
trend_step = ('trend', TrendTransformer(gold_series, exp_series, dota_adv['match_id']), ['match_id'])
current_transformers.append(trend_step)
pipeline.named_steps['preprocessor'].transformers = current_transformers

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
import optuna 

def objective(trial):
    global pipeline, X_train, X_val, y_train, y_val

    params = {
        'C': trial.suggest_float('C', 1e-5, 1000, log=True),
        'penalty': trial.suggest_categorical('penalty', ['l2']),
        'solver': 'lbfgs',
        'max_iter':  trial.suggest_int('max_iter', 1000, 2000, 100),
    }

    pipeline.set_params(classifier__C=params['C'], classifier__penalty=params['penalty'],
                        classifier__solver=params['solver'], classifier__max_iter=params['max_iter'])
    
    pipeline.fit(X_train.loc[:50000], y_train.loc[:50000])
    y_pred = pipeline.predict_proba(X_val)[:, 1]
    score = gini(y_val, y_pred)
    return score

study = optuna.create_study(direction='maximize')
study.optimize(objective, show_progress_bar=True, n_trials=25)

In [15]:
y_pred_test = pipeline.predict_proba(matches_df_test)[:, 1]
test_predictions = pd.DataFrame({
    'ID': matches_df_test['match_id'],
    'Value': y_pred_test
})
test_predictions.to_csv('../data/final/advanced_test_predictions.csv', index=False)

**New score**: 0.37503

Скор чуть вырос, то есть фича работает

## Заключение

Сохраняем пайплайн

In [17]:
joblib.dump(pipeline, '../models/advanced_pipeline.pkl')

['../models/advanced_pipeline.pkl']